# 04 — API walkthrough: list Zarr stores and metadata

Calls a **running** GeoAICubes API (`make api`, Compose, or any deployment).

| Method | Path |
|--------|------|
| `GET` | `/api/v1/inventory/zarr` |
| `GET` | `/api/v1/inventory/zarr/{id}` |

Configure the base URL (no trailing slash):

```bash
export GEOAICUBES_API_BASE_URL=http://localhost:8000
```


In [30]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd


## API base URL

Override in the next cell if your server is not on localhost.

In [31]:
API_BASE = os.environ.get("GEOAICUBES_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_BASE


'http://localhost:8000'

In [32]:
def get_json(path: str) -> tuple[int, dict]:
    url = f"{API_BASE}{path}"
    req = Request(url, headers={"Accept": "application/json"})
    with urlopen(req, timeout=120) as resp:
        body = json.loads(resp.read().decode())
        return resp.status, body


## Health check

In [33]:
status, health = get_json("/health")
status, health

(200, {'status': 'ok', 'service': 'geoaicubes-api', 'version': '0.1.0'})

## List all Zarr stores

In [34]:
status, data = get_json("/api/v1/inventory/zarr")
status, data["count"]

(200, 6)

In [35]:
pd.DataFrame(
    [{"id": s["id"], "status": s["status"], "crs": s.get("crs")} for s in data["stores"]]
)


,id,status,crs
0,GeoDataCube_cobertes-sol-v1r0-2023.zarr,ok,EPSG:25831
1,Geodatacube_S22024_2025.zarr,ok,EPSG:32631
2,Geodatacube_nDSM2024.zarr,ok,EPSG:25831
3,Geodatacube_nDSM2025.zarr,ok,EPSG:25831
4,Geodatacube_orto2024.zarr,ok,EPSG:25831
5,Geodatacube_orto2025.zarr,ok,EPSG:25831


## Single-store detail

In [36]:
store_id = data["stores"][0]["id"]
status, record = get_json(f"/api/v1/inventory/zarr/{quote(store_id)}")
status, store_id


(200, 'GeoDataCube_cobertes-sol-v1r0-2023.zarr')

In [37]:
print(json.dumps({k: record[k] for k in ("id", "status", "dims", "crs", "extent", "time")}, indent=2))
record["variables"]


{
  "id": "GeoDataCube_cobertes-sol-v1r0-2023.zarr",
  "status": "ok",
  "dims": [
    "y",
    "x"
  ],
  "crs": "EPSG:25831",
  "extent": {
    "bbox": [
      427000.5,
      4611300.5,
      458999.5,
      4630899.5
    ],
    "crs": "EPSG:25831"
  },
  "time": null
}


[{'name': 'data', 'dtype': 'uint8', 'shape': [19600, 32000]}]

## Unknown store → 404

In [38]:
try:
    get_json("/api/v1/inventory/zarr/not-a-store.zarr")
except HTTPError as exc:
    print(exc.code, json.loads(exc.read().decode()))


404 {'detail': 'Unknown Zarr store: not-a-store.zarr'}


Shell equivalents:

```bash
curl -s "$GEOAICUBES_API_BASE_URL/health" | jq
curl -s "$GEOAICUBES_API_BASE_URL/api/v1/inventory/zarr" | jq '.count'
```

Schema: `docs/data-plane/inventory-schema.md`.